# Experiment 5 — Comprehensive Study of CNN Training, Regularization, Optimization, Hyperparameter Tuning, Transfer Learning and Cross-Validation
**Model:** MobileNetV2 (ImageNet pretrained) &nbsp;|&nbsp; **Dataset:** Oxford-IIIT Pet (37 breeds, 224×224×3)

This notebook is organised to exactly mirror the sections of the lab manual (Sections 4–13, Plots 1–15).
Run the cells top to bottom. Each experiment cell trains quickly because only a small classifier head
sits on top of a *frozen* MobileNetV2 base (except in the fine-tuning section, Section 10, Case B).

**Important preprocessing note (a known pitfall from earlier experiments):** `tf.keras.applications.mobilenet_v2.preprocess_input`
expects raw pixel values in **[0, 255]**. Never divide by 255 first — if you do, you'll double-normalize and the model
will collapse (this happened with VGG16's `preprocess_input` in a previous experiment). Below we keep images as
raw `[0,255]` floats and let `preprocess_input` do the scaling to `[-1, 1]` itself.


In [ ]:
# --- Section 3/4 setup: imports ---
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, regularizers, initializers
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

tf.random.set_seed(42)
np.random.seed(42)

IMG_SIZE = 224
NUM_CLASSES = 37
AUTOTUNE = tf.data.AUTOTUNE


In [ ]:
# --- Section 3: Load Oxford-IIIT Pet dataset (direct download, no tensorflow_datasets) ---
# tensorflow_datasets pulls in tensorflow_metadata, which frequently hits protobuf
# gencode/runtime mismatches in Colab. We sidestep that entirely by downloading the
# official tarballs and building the pipeline ourselves.

import os, re, pathlib, tarfile, urllib.request
from PIL import Image

DATA_DIR = pathlib.Path('/content/oxford_pets')
DATA_DIR.mkdir(parents=True, exist_ok=True)

IMAGES_URL = 'https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz'
ANNOT_URL  = 'https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz'

def download_and_extract(url, dest_dir):
    fname = dest_dir / os.path.basename(url)
    if not fname.exists():
        print(f"Downloading {url} ...")
        urllib.request.urlretrieve(url, fname)
    print(f"Extracting {fname} ...")
    with tarfile.open(fname) as tar:
        tar.extractall(dest_dir)

download_and_extract(IMAGES_URL, DATA_DIR)
download_and_extract(ANNOT_URL, DATA_DIR)

IMAGES_DIR = DATA_DIR / 'images'
TRAINVAL_LIST = DATA_DIR / 'annotations' / 'trainval.txt'
TEST_LIST     = DATA_DIR / 'annotations' / 'test.txt'

def parse_list_file(path):
    # Each line: <image_name> <class_id> <species> <breed_id>
    filepaths, labels = [], []
    with open(path) as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            name, class_id, _, _ = line.strip().split(' ')
            img_path = IMAGES_DIR / f"{name}.jpg"
            if img_path.exists():
                filepaths.append(str(img_path))
                labels.append(int(class_id) - 1)   # make 0-indexed -> 0..36
    return filepaths, labels

trainval_paths, trainval_labels = parse_list_file(TRAINVAL_LIST)
test_paths, test_labels = parse_list_file(TEST_LIST)

# Breed names, derived from filenames (breed = filename with trailing _<number> stripped)
def breed_name_from_path(p):
    stem = pathlib.Path(p).stem
    return re.sub(r'_\d+$', '', stem).replace('_', ' ').title()

id_to_name = {}
for p, l in zip(trainval_paths, trainval_labels):
    id_to_name.setdefault(l, breed_name_from_path(p))
class_names = [id_to_name[i] for i in range(NUM_CLASSES)]

print(f"Train+val images: {len(trainval_paths)}, Test images: {len(test_paths)}, Classes: {len(class_names)}")

def preprocess_path(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)          # keep in [0,255] -- DO NOT rescale to [0,1] here
    image = preprocess_input(image)             # scales to [-1, 1] internally
    return image, label

BATCH = 32
VAL_FRACTION = 0.2

# Shuffle trainval together, then split into train/val
combined = list(zip(trainval_paths, trainval_labels))
rng = np.random.RandomState(42)
rng.shuffle(combined)
trainval_paths, trainval_labels = zip(*combined)
val_size = int(VAL_FRACTION * len(trainval_paths))

val_paths, val_labels     = list(trainval_paths[:val_size]), list(trainval_labels[:val_size])
train_paths, train_labels = list(trainval_paths[val_size:]), list(trainval_labels[val_size:])

def make_ds(paths, labels, shuffle=False, batch=BATCH):
    ds = tf.data.Dataset.from_tensor_slices((paths, list(labels)))
    ds = ds.map(preprocess_path, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(1000, seed=42)
    return ds.batch(batch).prefetch(AUTOTUNE)

train_ds = make_ds(train_paths, train_labels, shuffle=True)
val_ds   = make_ds(val_paths, val_labels)
test_ds  = make_ds(test_paths, test_labels)

print("Train batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Val batches:  ", tf.data.experimental.cardinality(val_ds).numpy())
print("Test batches: ", tf.data.experimental.cardinality(test_ds).numpy())


## Section 4 — MobileNetV2 Architecture
Loading the ImageNet-pretrained base and inspecting its structure / parameter count.


In [ ]:
base_model = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                          include_top=False,
                          weights='imagenet',
                          pooling='avg')
base_model.trainable = False   # frozen for feature extraction (Case A, Section 10)

print("Total params in base:", base_model.count_params())
base_model.summary()


## Section 5 — Weight Initialization
We attach a small trainable classifier head to the frozen MobileNetV2 base and compare
**Zero, Random Normal, Xavier/Glorot, He** initialization for the head's Dense layers.


In [ ]:
INIT_STRATEGIES = {
    "Zero":   initializers.Zeros(),
    "Random": initializers.RandomNormal(mean=0.0, stddev=0.05, seed=42),
    "Xavier": initializers.GlorotUniform(seed=42),
    "He":     initializers.HeNormal(seed=42),
}

def build_head(base, initializer, dropout=0.0, l2_reg=0.0, use_bn=False):
    reg = regularizers.l2(l2_reg) if l2_reg > 0 else None
    x = base.output
    x = layers.Dense(128, activation='relu', kernel_initializer=initializer, kernel_regularizer=reg)(x)
    if use_bn:
        x = layers.BatchNormalization()(x)
    if dropout > 0:
        x = layers.Dropout(dropout)(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax', kernel_initializer=initializer)(x)
    return models.Model(base.input, out)

EPOCHS_INIT = 6
init_histories = {}

for name, init in INIT_STRATEGIES.items():
    print(f"\n=== Training with {name} initialization ===")
    model = build_head(base_model, init)
    model.compile(optimizer=optimizers.Adam(1e-3),
                   loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_INIT, verbose=1)
    init_histories[name] = hist.history


In [ ]:
# Plot 1: Training Loss vs Epoch (per initialization)
plt.figure(figsize=(7,5))
for name, h in init_histories.items():
    plt.plot(range(1, EPOCHS_INIT+1), h['loss'], marker='o', label=name)
plt.xlabel('Epoch'); plt.ylabel('Training Loss')
plt.title('Plot 1: Training Loss vs Epoch (Weight Initialization)')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig('plot1_init_train_loss.png', dpi=150, bbox_inches='tight')
plt.show()

# Plot 2: Validation Accuracy vs Epoch (per initialization)
plt.figure(figsize=(7,5))
for name, h in init_histories.items():
    plt.plot(range(1, EPOCHS_INIT+1), [a*100 for a in h['val_accuracy']], marker='o', label=name)
plt.xlabel('Epoch'); plt.ylabel('Validation Accuracy (%)')
plt.title('Plot 2: Validation Accuracy vs Epoch (Weight Initialization)')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig('plot2_init_val_acc.png', dpi=150, bbox_inches='tight')
plt.show()

best_init_name = max(init_histories, key=lambda n: init_histories[n]['val_accuracy'][-1])
print("Best-performing initialization:", best_init_name)


## Section 6 — Regularization and Overfitting
Comparing **No Regularization, L2, Dropout, Batch Normalization** on the classifier head,
using the best initializer found in Section 5.


In [ ]:
BEST_INIT = INIT_STRATEGIES[best_init_name]
EPOCHS_REG = 8

REG_CONFIGS = {
    "No Regularization": dict(dropout=0.0, l2_reg=0.0, use_bn=False),
    "L2":                dict(dropout=0.0, l2_reg=1e-3, use_bn=False),
    "Dropout":           dict(dropout=0.5, l2_reg=0.0, use_bn=False),
    "BatchNorm":         dict(dropout=0.0, l2_reg=0.0, use_bn=True),
}

reg_histories = {}
for name, cfg in REG_CONFIGS.items():
    print(f"\n=== Training with {name} ===")
    model = build_head(base_model, BEST_INIT, **cfg)
    model.compile(optimizer=optimizers.Adam(1e-3),
                   loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_REG, verbose=1)
    reg_histories[name] = hist.history


In [ ]:
# Plot 3: Training & Validation Accuracy vs Epoch (one subplot per config)
fig, axes = plt.subplots(2, 2, figsize=(11,8))
for ax, (name, h) in zip(axes.ravel(), reg_histories.items()):
    ax.plot(range(1, EPOCHS_REG+1), [a*100 for a in h['accuracy']], label='Train Acc')
    ax.plot(range(1, EPOCHS_REG+1), [a*100 for a in h['val_accuracy']], label='Val Acc')
    ax.set_title(name); ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('Plot 3: Training & Validation Accuracy vs Epoch (Regularization)')
plt.tight_layout()
plt.savefig('plot3_reg_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

# Plot 4: Training & Validation Loss vs Epoch (one subplot per config)
fig, axes = plt.subplots(2, 2, figsize=(11,8))
for ax, (name, h) in zip(axes.ravel(), reg_histories.items()):
    ax.plot(range(1, EPOCHS_REG+1), h['loss'], label='Train Loss')
    ax.plot(range(1, EPOCHS_REG+1), h['val_loss'], label='Val Loss')
    ax.set_title(name); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('Plot 4: Training & Validation Loss vs Epoch (Regularization)')
plt.tight_layout()
plt.savefig('plot4_reg_loss.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 7 — Batch Normalization
### Numerical example (verifying the manual's worked example)


In [ ]:
x = np.array([2., 4., 6., 8.])
mu_B = x.mean()
var_B = x.var()          # population variance, matches the manual's 1/m formula
std_B = np.sqrt(var_B)
x_hat = (x - mu_B) / std_B

print("mu_B  =", mu_B)
print("var_B =", var_B)
print("std_B ~=", round(std_B, 3))
print("x_hat ~=", np.round(x_hat, 3))
# With gamma=1, beta=0 -> y = x_hat directly


In [ ]:
# Plot 5: With vs Without Batch Normalization -> Validation Accuracy vs Epoch
with_bn_hist    = reg_histories["BatchNorm"]
without_bn_hist = reg_histories["No Regularization"]

plt.figure(figsize=(7,5))
plt.plot(range(1, EPOCHS_REG+1), [a*100 for a in with_bn_hist['val_accuracy']], marker='o', label='With BN')
plt.plot(range(1, EPOCHS_REG+1), [a*100 for a in without_bn_hist['val_accuracy']], marker='o', label='Without BN')
plt.xlabel('Epoch'); plt.ylabel('Validation Accuracy (%)')
plt.title('Plot 5: With vs Without Batch Normalization')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig('plot5_bn_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best_reg_name = max(reg_histories, key=lambda n: reg_histories[n]['val_accuracy'][-1])
print("Best-performing regularization:", best_reg_name)


## Section 8 — Optimization Algorithms
Comparing **SGD, Momentum, RMSProp, Adam** using the best initializer + regularization found so far.


In [ ]:
import time

BEST_REG_CFG = REG_CONFIGS[best_reg_name]
EPOCHS_OPT = 6

OPTIMIZERS = {
    "SGD":      optimizers.SGD(learning_rate=1e-2),
    "Momentum": optimizers.SGD(learning_rate=1e-2, momentum=0.9),
    "RMSProp":  optimizers.RMSprop(learning_rate=1e-3),
    "Adam":     optimizers.Adam(learning_rate=1e-3),
}

opt_histories = {}
opt_table_rows = []

for name, opt in OPTIMIZERS.items():
    print(f"\n=== Training with {name} ===")
    model = build_head(base_model, BEST_INIT, **BEST_REG_CFG)
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    t0 = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_OPT, verbose=1)
    elapsed = time.time() - t0
    opt_histories[name] = hist.history

    best_val_acc = max(hist.history['val_accuracy'])
    epoch_to_converge = int(np.argmax(hist.history['val_accuracy'])) + 1
    opt_table_rows.append({
        "Optimizer": name,
        "Final Loss": round(hist.history['loss'][-1], 4),
        "Best Val. Accuracy": round(best_val_acc*100, 2),
        "Epoch to Converge": epoch_to_converge,
        "Time (s)": round(elapsed, 1)
    })

optimizer_table = pd.DataFrame(opt_table_rows)
print(optimizer_table)


In [ ]:
# Plot 6: Training Loss vs Epoch (per optimizer)
plt.figure(figsize=(7,5))
for name, h in opt_histories.items():
    plt.plot(range(1, EPOCHS_OPT+1), h['loss'], marker='o', label=name)
plt.xlabel('Epoch'); plt.ylabel('Training Loss')
plt.title('Plot 6: Training Loss vs Epoch (Optimizers)')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig('plot6_opt_train_loss.png', dpi=150, bbox_inches='tight')
plt.show()

# Plot 7: Validation Accuracy vs Epoch (per optimizer)
plt.figure(figsize=(7,5))
for name, h in opt_histories.items():
    plt.plot(range(1, EPOCHS_OPT+1), [a*100 for a in h['val_accuracy']], marker='o', label=name)
plt.xlabel('Epoch'); plt.ylabel('Validation Accuracy (%)')
plt.title('Plot 7: Validation Accuracy vs Epoch (Optimizers)')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig('plot7_opt_val_acc.png', dpi=150, bbox_inches='tight')
plt.show()

best_opt_name = optimizer_table.loc[optimizer_table['Best Val. Accuracy'].idxmax(), 'Optimizer']
print("Best-performing optimizer:", best_opt_name)


## Section 9 — CNN Hyperparameter Tuning
One-at-a-time sweep of **Learning Rate, Batch Size, Dropout Rate**, holding everything else at the
best configuration found so far (init = best_init, reg = best_reg, optimizer = best_opt).


In [ ]:
def build_and_train(lr=1e-3, batch_size=32, dropout=0.5, epochs=5, opt_name=None):
    opt_name = opt_name or best_opt_name
    if opt_name == "SGD":
        opt = optimizers.SGD(learning_rate=lr)
    elif opt_name == "Momentum":
        opt = optimizers.SGD(learning_rate=lr, momentum=0.9)
    elif opt_name == "RMSProp":
        opt = optimizers.RMSprop(learning_rate=lr)
    else:
        opt = optimizers.Adam(learning_rate=lr)

    tds = make_ds(raw_train_split, shuffle=True, batch=batch_size)
    vds = make_ds(raw_val, batch=batch_size)

    model = build_head(base_model, BEST_INIT, dropout=dropout, l2_reg=BEST_REG_CFG.get('l2_reg',0.0),
                        use_bn=BEST_REG_CFG.get('use_bn', False))
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    hist = model.fit(tds, validation_data=vds, epochs=epochs, verbose=0)
    return max(hist.history['val_accuracy']) * 100

# --- Learning Rate sweep ---
lr_values = [0.001, 0.0001]
lr_results = {lr: build_and_train(lr=lr) for lr in lr_values}

# --- Batch Size sweep ---
batch_values = [16, 32, 64]
batch_results = {b: build_and_train(batch_size=b) for b in batch_values}

# --- Dropout Rate sweep ---
dropout_values = [0.0, 0.25, 0.5]
dropout_results = {d: build_and_train(dropout=d) for d in dropout_values}

print("LR results:", lr_results)
print("Batch size results:", batch_results)
print("Dropout results:", dropout_results)


In [ ]:
# Plot 8: Learning Rate vs Validation Accuracy
plt.figure(figsize=(6,4))
plt.plot(list(map(str, lr_results.keys())), list(lr_results.values()), marker='o')
plt.xlabel('Learning Rate'); plt.ylabel('Validation Accuracy (%)')
plt.title('Plot 8: Learning Rate vs Validation Accuracy'); plt.grid(alpha=0.3)
plt.savefig('plot8_lr_vs_acc.png', dpi=150, bbox_inches='tight'); plt.show()

# Plot 9: Batch Size vs Validation Accuracy
plt.figure(figsize=(6,4))
plt.plot(list(map(str, batch_results.keys())), list(batch_results.values()), marker='o')
plt.xlabel('Batch Size'); plt.ylabel('Validation Accuracy (%)')
plt.title('Plot 9: Batch Size vs Validation Accuracy'); plt.grid(alpha=0.3)
plt.savefig('plot9_batch_vs_acc.png', dpi=150, bbox_inches='tight'); plt.show()

# Plot 10: Dropout Rate vs Validation Accuracy
plt.figure(figsize=(6,4))
plt.plot(list(map(str, dropout_results.keys())), list(dropout_results.values()), marker='o')
plt.xlabel('Dropout Rate'); plt.ylabel('Validation Accuracy (%)')
plt.title('Plot 10: Dropout Rate vs Validation Accuracy'); plt.grid(alpha=0.3)
plt.savefig('plot10_dropout_vs_acc.png', dpi=150, bbox_inches='tight'); plt.show()

best_lr = max(lr_results, key=lr_results.get)
best_batch = max(batch_results, key=batch_results.get)
best_dropout = max(dropout_results, key=dropout_results.get)
print(f"Best LR={best_lr}, Best batch size={best_batch}, Best dropout={best_dropout}")


## Section 10 — Transfer Learning and Fine-Tuning
**Case A (Feature Extraction):** base frozen, only classifier head trained (as above).
**Case B (Fine-Tuning):** unfreeze the top layers of MobileNetV2 and continue training with a much
smaller learning rate.


In [ ]:
EPOCHS_FE = 5
EPOCHS_FT = 5

# --- Case A: Feature Extraction (fresh run with the best config, for a clean curve) ---
base_model.trainable = False
model_fe = build_head(base_model, BEST_INIT, dropout=best_dropout,
                       l2_reg=BEST_REG_CFG.get('l2_reg', 0.0), use_bn=BEST_REG_CFG.get('use_bn', False))
model_fe.compile(optimizer=optimizers.Adam(best_lr),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist_fe = model_fe.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FE, verbose=1)

# --- Case B: Fine-Tuning -- unfreeze top N layers of the base, small LR ---
UNFREEZE_FROM = len(base_model.layers) - 30   # unfreeze last 30 layers
base_model.trainable = True
for layer in base_model.layers[:UNFREEZE_FROM]:
    layer.trainable = False

model_fe.compile(optimizer=optimizers.Adam(1e-5),   # smaller LR for fine-tuning
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist_ft = model_fe.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, verbose=1)


In [ ]:
# Plot 11: Feature Extraction vs Fine-Tuning -> Validation Accuracy vs Epoch
fe_val = [a*100 for a in hist_fe.history['val_accuracy']]
ft_val = [a*100 for a in hist_ft.history['val_accuracy']]

plt.figure(figsize=(7,5))
plt.plot(range(1, EPOCHS_FE+1), fe_val, marker='o', label='Feature Extraction')
plt.plot(range(EPOCHS_FE+1, EPOCHS_FE+EPOCHS_FT+1), ft_val, marker='o', label='Fine-Tuning')
plt.axvline(EPOCHS_FE + 0.5, color='gray', linestyle='--', alpha=0.6)
plt.xlabel('Epoch'); plt.ylabel('Validation Accuracy (%)')
plt.title('Plot 11: Feature Extraction vs Fine-Tuning')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig('plot11_fe_vs_ft.png', dpi=150, bbox_inches='tight')
plt.show()

# Plot 12: Training & Validation Loss before/after fine-tuning
fe_loss, fe_vloss = hist_fe.history['loss'], hist_fe.history['val_loss']
ft_loss, ft_vloss = hist_ft.history['loss'], hist_ft.history['val_loss']

plt.figure(figsize=(7,5))
plt.plot(range(1, EPOCHS_FE+1), fe_loss, marker='o', label='Train Loss (Feature Extraction)')
plt.plot(range(1, EPOCHS_FE+1), fe_vloss, marker='o', label='Val Loss (Feature Extraction)')
plt.plot(range(EPOCHS_FE+1, EPOCHS_FE+EPOCHS_FT+1), ft_loss, marker='s', label='Train Loss (Fine-Tuning)')
plt.plot(range(EPOCHS_FE+1, EPOCHS_FE+EPOCHS_FT+1), ft_vloss, marker='s', label='Val Loss (Fine-Tuning)')
plt.axvline(EPOCHS_FE + 0.5, color='gray', linestyle='--', alpha=0.6)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Plot 12: Training & Validation Loss (Before/After Fine-Tuning)')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig('plot12_ft_loss.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 11 — K-Fold Cross-Validation
Select 3–4 promising configurations (from the studies above) and evaluate each with 5-fold
stratified cross-validation on the **training data only**. The test set stays untouched.


In [ ]:
# Collect (image, label) arrays for the full train+val pool, for CV splitting
def collect_arrays(paths, labels):
    imgs = []
    for p in paths:
        image = tf.io.read_file(p)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
        imgs.append(image.numpy())
    return np.array(imgs, dtype=np.float32), np.array(labels)

# NOTE: this materializes the training pool in memory; fine for Oxford-IIIT Pet (~3.7k images).
pool_paths = list(train_paths) + list(val_paths)
pool_labels = list(train_labels) + list(val_labels)
X_pool, y_pool = collect_arrays(pool_paths, pool_labels)
print("Pool shape:", X_pool.shape, y_pool.shape)

CV_CONFIGS = {
    "C1_Baseline":        dict(dropout=0.0, l2_reg=0.0, use_bn=False, lr=1e-3, opt='Adam'),
    "C2_BestReg":         dict(dropout=best_dropout, l2_reg=BEST_REG_CFG.get('l2_reg',0.0),
                                use_bn=BEST_REG_CFG.get('use_bn', False), lr=best_lr, opt=best_opt_name),
    "C3_HighDropout":     dict(dropout=0.5, l2_reg=0.0, use_bn=False, lr=1e-3, opt='Adam'),
    "C4_L2_BN":           dict(dropout=0.0, l2_reg=1e-3, use_bn=True, lr=1e-3, opt='Adam'),
}

EPOCHS_CV = 5
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {name: [] for name in CV_CONFIGS}

for name, cfg in CV_CONFIGS.items():
    print(f"\n=== 5-Fold CV: {name} ===")
    fold_accs = []
    for fold_i, (tr_idx, va_idx) in enumerate(skf.split(X_pool, y_pool), start=1):
        X_tr = preprocess_input(X_pool[tr_idx].copy())
        X_va = preprocess_input(X_pool[va_idx].copy())
        y_tr, y_va = y_pool[tr_idx], y_pool[va_idx]

        if cfg['opt'] == 'SGD':
            opt = optimizers.SGD(learning_rate=cfg['lr'])
        elif cfg['opt'] == 'Momentum':
            opt = optimizers.SGD(learning_rate=cfg['lr'], momentum=0.9)
        elif cfg['opt'] == 'RMSProp':
            opt = optimizers.RMSprop(learning_rate=cfg['lr'])
        else:
            opt = optimizers.Adam(learning_rate=cfg['lr'])

        m = build_head(base_model, BEST_INIT, dropout=cfg['dropout'],
                        l2_reg=cfg['l2_reg'], use_bn=cfg['use_bn'])
        m.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        m.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=EPOCHS_CV, batch_size=32, verbose=0)
        val_acc = m.evaluate(X_va, y_va, verbose=0)[1]
        fold_accs.append(val_acc * 100)
        print(f"  Fold {fold_i}: {val_acc*100:.2f}%")
    cv_results[name] = fold_accs

cv_table = pd.DataFrame({
    name: accs for name, accs in cv_results.items()
}).T
cv_table.columns = [f"F{i}" for i in range(1,6)]
cv_table["Mean"] = cv_table.mean(axis=1)
cv_table["SD"] = cv_table[[f"F{i}" for i in range(1,6)]].std(axis=1)
print(cv_table)


In [ ]:
# Plot 13: 5-Fold CV Accuracy per configuration, with SD error bars
plt.figure(figsize=(7,5))
plt.bar(cv_table.index, cv_table['Mean'], yerr=cv_table['SD'], capsize=6, alpha=0.8)
plt.xlabel('Hyperparameter Configuration'); plt.ylabel('Mean Validation Accuracy (%)')
plt.title('Plot 13: 5-Fold Cross-Validation Accuracy')
plt.grid(alpha=0.3, axis='y')
plt.savefig('plot13_kfold_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

best_cv_config_name = cv_table['Mean'].idxmax()
print("Selected configuration:", best_cv_config_name)


## Section 12 — Final Model Evaluation
Retrain the selected configuration on the **complete training data** (train+val pool), then evaluate
**once** on the untouched test set.


In [ ]:
best_cfg = CV_CONFIGS[best_cv_config_name]

if best_cfg['opt'] == 'SGD':
    final_opt = optimizers.SGD(learning_rate=best_cfg['lr'])
elif best_cfg['opt'] == 'Momentum':
    final_opt = optimizers.SGD(learning_rate=best_cfg['lr'], momentum=0.9)
elif best_cfg['opt'] == 'RMSProp':
    final_opt = optimizers.RMSprop(learning_rate=best_cfg['lr'])
else:
    final_opt = optimizers.Adam(learning_rate=best_cfg['lr'])

final_model = build_head(base_model, BEST_INIT, dropout=best_cfg['dropout'],
                          l2_reg=best_cfg['l2_reg'], use_bn=best_cfg['use_bn'])
final_model.compile(optimizer=final_opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

import time
t0 = time.time()
final_history = final_model.fit(train_ds.concatenate(val_ds), epochs=10, verbose=1)
training_time = time.time() - t0

test_loss, test_acc = final_model.evaluate(test_ds, verbose=1)
print(f"Test Accuracy: {test_acc*100:.2f}%")


In [ ]:
# Collect predictions on the test set for Precision/Recall/F1/Confusion Matrix
y_true, y_pred, test_images = [], [], []
for images, labels in test_ds:
    preds = final_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    test_images.extend(images.numpy())

y_true = np.array(y_true); y_pred = np.array(y_pred)

precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall    = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1        = f1_score(y_true, y_pred, average='macro', zero_division=0)
num_params = final_model.count_params()

final_metrics = {
    "Mean CV Accuracy": round(cv_table.loc[best_cv_config_name, 'Mean'], 2),
    "CV Standard Deviation": round(cv_table.loc[best_cv_config_name, 'SD'], 2),
    "Test Accuracy": round(test_acc*100, 2),
    "Precision": round(precision, 4),
    "Recall": round(recall, 4),
    "F1-score": round(f1, 4),
    "Training Time (s)": round(training_time, 1),
    "Number of Parameters": num_params
}
print(pd.Series(final_metrics))


In [ ]:
# Plot 14: Confusion Matrix
# class_names already defined in Section 3
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(14,12))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks(range(NUM_CLASSES), class_names, rotation=90, fontsize=6)
plt.yticks(range(NUM_CLASSES), class_names, fontsize=6)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Plot 14: Confusion Matrix')
plt.tight_layout()
plt.savefig('plot14_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Identify best-classified and most-confused classes
diag = np.diag(cm)
per_class_acc = diag / cm.sum(axis=1)
best_classes = np.argsort(per_class_acc)[::-1][:5]
print("Best classified classes:", [class_names[i] for i in best_classes])

cm_no_diag = cm.copy().astype(float)
np.fill_diagonal(cm_no_diag, 0)
i_true, i_pred = np.unravel_index(np.argmax(cm_no_diag), cm_no_diag.shape)
print(f"Most confused pair: true='{class_names[i_true]}' predicted as '{class_names[i_pred]}' ({int(cm_no_diag[i_true, i_pred])} times)")


In [ ]:
# Optional Plot 15: Misclassified Images
mis_idx = np.where(y_true != y_pred)[0][:6]

fig, axes = plt.subplots(2, 3, figsize=(12,8))
for ax, idx in zip(axes.ravel(), mis_idx):
    img = (test_images[idx] + 1) / 2.0   # undo preprocess_input's [-1,1] scaling for display
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(f"True: {class_names[y_true[idx]]}\nPred: {class_names[y_pred[idx]]}", fontsize=8)
    ax.axis('off')
plt.suptitle('Plot 15 (Optional): Misclassified Images')
plt.tight_layout()
plt.savefig('plot15_misclassified.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 13 — Overall Results
Fill this table in from the metrics/plots produced above (Baseline = `No Regularization` config
from Section 6 trained with default Adam, LR=1e-3).


In [ ]:
overall_results = pd.DataFrame([
    {"Configuration": "Baseline",              "CV Accuracy": np.nan, "SD": np.nan,
     "Test Accuracy": np.nan, "Training Time": np.nan},
    {"Configuration": "Best Initialization",    "CV Accuracy": np.nan, "SD": np.nan,
     "Test Accuracy": np.nan, "Training Time": np.nan},
    {"Configuration": "Best Regularization",    "CV Accuracy": np.nan, "SD": np.nan,
     "Test Accuracy": np.nan, "Training Time": np.nan},
    {"Configuration": "Best Optimizer",         "CV Accuracy": np.nan, "SD": np.nan,
     "Test Accuracy": np.nan, "Training Time": np.nan},
    {"Configuration": "Best Hyperparameters",   "CV Accuracy": np.nan, "SD": np.nan,
     "Test Accuracy": np.nan, "Training Time": np.nan},
    {"Configuration": "Fine-Tuned Model", "CV Accuracy": cv_table.loc[best_cv_config_name,'Mean'],
     "SD": cv_table.loc[best_cv_config_name,'SD'], "Test Accuracy": test_acc*100,
     "Training Time": training_time},
])
overall_results


## Section 16 — Additional Exercise
Select **two new combinations** of learning rate, dropout, batch size and fine-tuning strategy, run
them through the same `build_and_train` / 5-fold-CV machinery above, and compare against
`best_cv_config_name`. Justify preference using accuracy, SD, cost and test performance.


In [ ]:
# Example scaffold -- edit the two configs below and re-use the CV loop pattern from Section 11
EXTRA_CONFIGS = {
    "New_A": dict(dropout=0.3, l2_reg=0.0, use_bn=False, lr=5e-4, opt='Adam'),
    "New_B": dict(dropout=0.4, l2_reg=1e-4, use_bn=True,  lr=1e-3, opt='RMSProp'),
}
# Re-run the Section 11 CV loop with EXTRA_CONFIGS in place of CV_CONFIGS to populate this section.
